In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel


In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 16
LEARNING_RATE = 2e-5

PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("higopires/RePro-categories-multilabel")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df


,review_text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,INADEQUADA,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0,0
...,...,...,...,...,...,...,...
7997,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0,0
7998,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0,0
7999,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0,0
8000,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0,0


In [4]:
train_df = train_df[train_df['INADEQUADA'] == 0].reset_index(drop=True)
val_df = val_df[val_df['INADEQUADA'] == 0].reset_index(drop=True)
test_df = test_df[test_df['INADEQUADA'] == 0].reset_index(drop=True)

train_df = train_df.drop(columns=['INADEQUADA'])
val_df = val_df.drop(columns=['INADEQUADA'])
test_df = test_df.drop(columns=['INADEQUADA'])

train_df = train_df.rename(columns={'review_text': 'text'})
val_df = val_df.rename(columns={'review_text': 'text'})
test_df = test_df.rename(columns={'review_text': 'text'})

train_df

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0
...,...,...,...,...,...,...
7669,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0
7670,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0
7671,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0
7672,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0


In [5]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., BertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [6]:
class BertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiLabelClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [7]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [8]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [9]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [10]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

num_classes = train_labels.shape[1]

# Create datasets
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
results = []

for seed in seeds:
    torch.manual_seed(seed)
    model = BertForMultiLabelClassification(num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label classification

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    save_path = f'results/bert_multilabel1_bs{BATCH_SIZE}_lr{LEARNING_RATE}_seed{seed}.pt'

    # Train
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
         {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval

    # Load best model
    model.load_state_dict(torch.load(save_path))

    # Evaluate
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, test_retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = test_retval
    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    # Store individual seed results
    results.append({
        'seed': seed,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_acc': train_acc,
        'train_precisions': train_precisions.tolist(),
        'train_recalls': train_recalls.tolist(),
        'train_f1s': train_f1s.tolist(),
        'max_memory_usage_train': max_memory_usage_train,
        'max_vram_usage_train': max_vram_usage_train,
        'total_train_time': total_train_time,
        'val_acc': val_acc,
        'val_precisions': val_precisions.tolist(),
        'val_recalls': val_recalls.tolist(),
        'val_f1s': val_f1s.tolist(),
        'test_acc': test_acc,
        'test_precisions': test_precisions.tolist(),
        'test_recalls': test_recalls.tolist(),
        'test_f1s': test_f1s.tolist(),
        'max_memory_usage_test': max_memory_usage_test,
        'max_vram_usage_test': max_vram_usage_test,
        'total_test_time': total_time_test
    })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/480 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:407: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/4 - Train Loss: 0.3633, Acc: 0.5713, F1: [0.75358852 0.46270352 0.9044922  0.3039807  0.01149425]
Epoch 1/4 - Val Loss: 0.2515, Acc: 0.6607, F1: [0.91036907 0.70050761 0.94484101 0.54782609 0.        ]
Model saved!


Epoch 2/4 - Train Loss: 0.2199, Acc: 0.7032, F1: [0.91434644 0.76130578 0.93834611 0.72115794 0.38028169]
Epoch 2/4 - Val Loss: 0.1887, Acc: 0.7185, F1: [0.93333333 0.8097561  0.93341554 0.65863454 0.74074074]
Model saved!


Epoch 3/4 - Train Loss: 0.1546, Acc: 0.7733, F1: [0.94439764 0.83528722 0.95209435 0.81245012 0.77683135]
Epoch 3/4 - Val Loss: 0.1559, Acc: 0.7794, F1: [0.95769882 0.8287037  0.95624196 0.76428571 0.83221477]
Model saved!


Epoch 4/4 - Train Loss: 0.1168, Acc: 0.8236, F1: [0.96200929 0.87745665 0.96305399 0.86224292 0.85167464]
Epoch 4/4 - Val Loss: 0.1515, Acc: 0.7763, F1: [0.9544688  0.82100239 0.95702373 0.80634921 0.85365854]
Model saved!


...

Epoch 2/4 - Train Loss: 0.2099, Acc: 0.7194, F1: [0.91705459 0.77727683 0.94103334 0.70477816 0.66666667]
Epoch 2/4 - Val Loss: 0.1730, Acc: 0.7437, F1: [0.9375     0.78801843 0.94429708 0.7628866  0.83435583]
Model saved!


Epoch 3/4 - Train Loss: 0.1516, Acc: 0.7820, F1: [0.94958869 0.8410943  0.95308038 0.80497592 0.80741338]
Epoch 3/4 - Val Loss: 0.1639, Acc: 0.7689, F1: [0.9375     0.82837529 0.95623775 0.77702703 0.78873239]
Model saved!


Epoch 4/4 - Train Loss: 0.1206, Acc: 0.8208, F1: [0.9652412  0.87840185 0.96122633 0.84777518 0.85669291]
Epoch 4/4 - Val Loss: 0.1530, Acc: 0.7763, F1: [0.95918367 0.80871671 0.95579757 0.79354839 0.8516129 ]
Model saved!


C:\Users\Rafael\AppData\Local\Temp\ipykernel_7804\357557947.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafael

Test Time: 8.27 seconds
Test Metrics:
Accuracy: 0.7805383022774327
F1s: [0.94472362 0.81372549 0.94642857 0.8189911  0.84848485]
Precisions: [0.94314381 0.87830688 0.91945477 0.79768786 0.86419753]
Recalls: [0.94630872 0.75799087 0.97503285 0.84146341 0.83333333]


In [11]:
df = pd.DataFrame(results)
df.to_csv('results/bert_multilabel1.csv', index=False)

In [12]:
df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,16,0.00002,0.823560,"[0.9685507862303443, 0.8830715532286213, 0.961...","[0.9555555555555556, 0.8719126938541069, 0.964...","[0.9620092866188266, 0.8774566473988439, 0.963...",1269.371094,2528.906738,340.161490,...,"[0.9593220338983051, 0.8472906403940886, 0.936...","[0.9496644295302014, 0.7962962962962963, 0.979...","[0.954468802698145, 0.8210023866348448, 0.9570...",0.770186,"[0.9494949494949495, 0.8730964467005076, 0.929...","[0.9463087248322147, 0.7853881278538812, 0.971...","[0.9478991596638655, 0.8269230769230769, 0.949...",1286.820312,1704.052734,8.760536
1,3,16,0.00002,0.774433,"[0.9566694987255735, 0.8620481927710844, 0.940...","[0.9442348008385745, 0.8219414129810454, 0.958...","[0.9504114792150242, 0.8415172008232873, 0.949...",1290.125000,2540.281738,342.954207,...,"[0.9437086092715232, 0.8571428571428571, 0.926...","[0.9563758389261745, 0.8055555555555556, 0.982...","[0.95, 0.8305489260143198, 0.954140127388535, ...",0.781573,"[0.9153094462540716, 0.8823529411764706, 0.934...","[0.9429530201342282, 0.821917808219178, 0.9671...","[0.9289256198347108, 0.851063829787234, 0.9502...",1209.468750,1706.427734,8.750499
2,5,16,0.00002,0.820824,"[0.9699407281964437, 0.8855808523058961, 0.960...","[0.9605870020964361, 0.871338311315336, 0.9622...","[0.965241204971561, 0.878401852924146, 0.96122...",1290.882812,2536.906738,339.363477,...,"[0.9724137931034482, 0.8477157360406091, 0.933...","[0.9463087248322147, 0.7731481481481481, 0.979...","[0.9591836734693877, 0.8087167070217918, 0.955...",0.780538,"[0.9431438127090301, 0.8783068783068783, 0.919...","[0.9463087248322147, 0.7579908675799086, 0.975...","[0.9447236180904522, 0.8137254901960784, 0.946...",1290.035156,1705.427734,8.759381
